# G-EMO Android APK Build — Google Colab

This notebook builds the `implementation/g-emo-core` branch of `SE-Emam/G-EMO`.

**Changes in this version**
- Build output is written to a full log instead of flooding the notebook output, reducing browser UI freezes.
- Progress is printed every 30 seconds.
- A per-attempt timeout stops a build that exceeds 90 minutes; no claim of success is made without an APK.
- The known pip/python-for-android import mismatch triggers only a narrow source workaround when the exact error appears.
- APK existence, size, SHA-256, and download are checked at the end.

A successful build does not replace an install/launch test on an Android device.

In [ ]:
# 1) Install Linux/Android build prerequisites
!apt-get -qq update
!apt-get -qq install -y git zip unzip openjdk-17-jdk python3-pip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo6 cmake libffi-dev libssl-dev automake
%env JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
%env ANDROID_ACCEPT_SDK_LICENSE=1
%env ACCEPT_ANDROID_SDK_LICENSE=1
!java -version
!python --version

In [ ]:
# 2) Clone the exact implementation branch and install build tools
%cd /content
!rm -rf G-EMO
!git clone --depth 1 --branch implementation/g-emo-core https://github.com/SE-Emam/G-EMO.git G-EMO
%cd /content/G-EMO
!git rev-parse HEAD
!test -f buildozer.spec && sed -n '1,160p' buildozer.spec
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install buildozer 'Cython==0.29.36'
!buildozer --version

In [ ]:
# 3) Build with a quiet console, complete log, timeout, and conditional retry
import os, sys, time, shutil, subprocess, hashlib
from pathlib import Path

PROJECT = Path("/content/G-EMO")
LOG_PATH = Path("/content/g-emo-build.log")
POLL_SECONDS = 30
BUILD_TIMEOUT_SECONDS = 90 * 60  # 90 minutes per attempt

def section(title):
    print("\
" + "=" * 72 + "\
" + title + "\
" + "=" * 72, flush=True)

def tail_text(path, count=100):
    try:
        return "\
".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-count:])
    except Exception as exc:
        return f"Could not read log: {exc}"

def run_build(label, log_mode="w"):
    section(label)
    started = time.monotonic()
    last_size = -1
    last_growth = started
    timed_out = False

    with LOG_PATH.open(log_mode, encoding="utf-8", errors="replace") as log:
        log.write(f"\
\
===== {label} =====\
")
        log.flush()
        proc = subprocess.Popen(
            ["buildozer", "-v", "android", "debug"],
            cwd=str(PROJECT),
            stdout=log,
            stderr=subprocess.STDOUT,
        )

        while proc.poll() is None:
            time.sleep(POLL_SECONDS)
            elapsed = int(time.monotonic() - started)
            size = LOG_PATH.stat().st_size if LOG_PATH.exists() else 0

            if size != last_size:
                last_size = size
                last_growth = time.monotonic()

            try:
                with LOG_PATH.open("rb") as f:
                    f.seek(max(0, size - 12000))
                    tail = f.read().decode("utf-8", errors="replace")
                latest = next((s.strip() for s in reversed(tail.splitlines()) if s.strip()),
                              "Waiting for build output...")
            except Exception:
                latest = "Waiting for build output..."

            stagnant = int(time.monotonic() - last_growth)
            print(
                f"Elapsed {elapsed//60}m {elapsed%60:02d}s | "
                f"log {size/1024:.1f} KiB | no log growth {stagnant}s\
"
                f"Latest: {latest[-220:]}",
                flush=True,
            )

            if elapsed >= BUILD_TIMEOUT_SECONDS:
                timed_out = True
                print("TIMEOUT: stopping this build attempt; full log is preserved.", flush=True)
                proc.terminate()
                try:
                    proc.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    proc.kill()
                    proc.wait()
                break

        rc = proc.wait()
        log.write(f"\
===== EXIT CODE: {rc}; TIMED_OUT={timed_out} =====\
")
        log.flush()

    print(f"Exit code: {rc}; timed out: {timed_out}", flush=True)
    return rc, timed_out

# Validate paths/tools before spending time on a build.
section("Preflight checks")
if not PROJECT.is_dir():
    raise FileNotFoundError(f"Project not found: {PROJECT}. Run the clone cell first.")
if not (PROJECT / "buildozer.spec").is_file():
    raise FileNotFoundError(f"Missing {PROJECT / 'buildozer.spec'}; wrong project root.")
if shutil.which("buildozer") is None:
    raise RuntimeError("Buildozer is not installed. Run the setup cells first.")
print("PASS: project, buildozer.spec, and Buildozer are present.")
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip())

# First attempt: all output goes to disk, not the notebook output stream.
rc, timed_out = run_build("Buildozer first attempt", "w")

# Retry only when the exact known import mismatch is found.
if rc != 0 and not timed_out:
    log_text = LOG_PATH.read_text(encoding="utf-8", errors="replace")
    known_error = (
        "BuildDependencyInstallError" in log_text
        and "pip._internal.exceptions" in log_text
    )

    if known_error:
        section("Known pip/python-for-android mismatch detected")
        p4a_root = PROJECT / ".buildozer/android/platform/python-for-android"
        changed = []

        if p4a_root.is_dir():
            for file in p4a_root.rglob("*.py"):
                try:
                    source = file.read_text(encoding="utf-8")
                except (OSError, UnicodeError):
                    continue
                replacement = source.replace(
                    "pip install -U pip",
                    "pip install --upgrade --force-reinstall pip",
                )
                if replacement != source:
                    file.write_text(replacement, encoding="utf-8")
                    changed.append(str(file))

        print("Patched files:", changed if changed else "NONE")
        if changed:
            rc, timed_out = run_build("Buildozer retry after narrow pip workaround", "a")
        else:
            print("No exact command found; no files changed and no blind retry performed.")
    else:
        print("Failure differs from the known pip mismatch. No automatic patch applied.")

# Show bounded diagnostics without flooding the browser.
section("Last 100 log lines")
print(tail_text(LOG_PATH, 100), flush=True)

# Verify output; an exit code alone is not enough.
section("APK verification")
apk_dir = PROJECT / "bin"
apks = sorted(apk_dir.rglob("*.apk"), key=lambda p: p.stat().st_mtime, reverse=True) if apk_dir.is_dir() else []

if rc == 0 and apks:
    print(f"PASS: found {len(apks)} APK file(s).")
    for apk in apks:
        digest = hashlib.sha256(apk.read_bytes()).hexdigest()
        print(f"APK: {apk}\
Size: {apk.stat().st_size:,} bytes\
SHA-256: {digest}\
")
else:
    print("FAIL/BLOCKED: no verified successful APK build.")
    print("Build exit code:", rc, "| timed out:", timed_out)

# Download APK only after the existence check; otherwise offer the log.
try:
    from google.colab import files
    if rc == 0 and apks:
        chosen = apks[0]
        print("Starting APK download:", chosen, flush=True)
        files.download(str(chosen))
    else:
        print("Download the diagnostic log from the Colab Files panel:", LOG_PATH)
except Exception as exc:
    print("Colab download helper unavailable:", repr(exc))
    print("Files to inspect:", (apks[0] if apks else LOG_PATH))